In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# !rm -rf VDD_ComparativeAnalysis
# !git clone https://github.com/pri-13/VDD_ComparativeAnalysis.git
# %cd VDD_ComparativeAnalysis

Cloning into 'VDD_ComparativeAnalysis'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 58 (delta 22), reused 33 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 8.18 MiB | 15.55 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/VDD_ComparativeAnalysis


In [4]:
# !pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 778.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/1

In [4]:
#imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

from src.config import *
from src.dataset import create_dataset, optimize_dataset

In [5]:
#dataset path
DATA_DIR = Path("/content/drive/MyDrive/VDD_Dataset")
PROCESSED_DATA_DIR = DATA_DIR / "processed_data"
TEST_DIR = PROCESSED_DATA_DIR / "test"
SAVED_MODELS_DIR = DATA_DIR / "saved_models"
EVALUATION_OUTPUT_DIR = (DATA_DIR / "outputs" / "evaluation")
EVALUATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
#load dataset
test_dataset = create_dataset(TEST_DIR, shuffle=False)
CLASS_LABELS = test_dataset.class_names
test_dataset = optimize_dataset(test_dataset)
print(CLASS_LABELS)

Found 555 files belonging to 6 classes.
['crack', 'dent', 'glass shatter', 'lamp broken', 'scratch', 'tire flat']


In [7]:
#models
MODELS = ["custom_cnn", "vgg16", "mobilenetv2", "densenet121"]

In [8]:
#evaluation function
def evaluate_model(model_name):
    print(f"Evaluating {model_name}")
    model = tf.keras.models.load_model(SAVED_MODELS_DIR / f"{model_name}.keras")
    loss, accuracy = model.evaluate(test_dataset, verbose=0)
    predictions = model.predict(test_dataset, verbose=0)
    y_pred = np.argmax(predictions, axis=1)
    y_true = np.concatenate([labels.numpy() for _, labels in test_dataset])
    precision = precision_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )
    recall = recall_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )
    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )
    balanced = balanced_accuracy_score(
        y_true,
        y_pred
    )
    report = classification_report(
        y_true,
        y_pred,
        target_names=CLASS_LABELS,
        output_dict=True,
        zero_division=0
    )
    pd.DataFrame(report).transpose().to_csv(EVALUATION_OUTPUT_DIR / f"{model_name}_classification_report.csv")
    cm = confusion_matrix(
        y_true,
        y_pred
    )
    fig, ax = plt.subplots(figsize=(8,8))
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=CLASS_LABELS
    ).plot(
        ax=ax,
        xticks_rotation=45,
        colorbar=False
    )
    plt.title(model_name)
    plt.tight_layout()
    plt.savefig(
        EVALUATION_OUTPUT_DIR /
        f"{model_name}_confusion_matrix.png",
        dpi=300
    )
    plt.close()
    tf.keras.backend.clear_session()
    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Balanced Accuracy": balanced,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Loss": loss
    }

In [9]:
#evaluate all models
results = []
for model_name in MODELS:
    metrics = evaluate_model(model_name)
    results.append(metrics)
results_df = pd.DataFrame(results)
results_df

Evaluating custom_cnn
Evaluating vgg16
Evaluating mobilenetv2
Evaluating densenet121


,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1 Score,Loss
0,custom_cnn,0.477477,0.425349,0.440118,0.477477,0.445059,1.288558
1,vgg16,0.515315,0.502434,0.510320,0.515315,0.507838,2.053815
2,mobilenetv2,0.563964,0.519527,0.557603,0.563964,0.543237,1.090534
3,densenet121,0.535135,0.483689,0.526089,0.535135,0.514955,1.099862


In [10]:
#save results
results_df.to_csv(EVALUATION_OUTPUT_DIR / "evaluation_results.csv", index=False)
results_df

,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1 Score,Loss
0,custom_cnn,0.477477,0.425349,0.440118,0.477477,0.445059,1.288558
1,vgg16,0.515315,0.502434,0.510320,0.515315,0.507838,2.053815
2,mobilenetv2,0.563964,0.519527,0.557603,0.563964,0.543237,1.090534
3,densenet121,0.535135,0.483689,0.526089,0.535135,0.514955,1.099862
